# Exploratory Data Analysis - KDDCup'99 Dataset
## Network Anomaly Detection Project

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_kddcup99
from sklearn.preprocessing import LabelEncoder

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

## 1. Load Dataset

In [ ]:
COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate",
    "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "label"
]

kdd = fetch_kddcup99(as_frame=True)
df = kdd.frame
df.columns = COLUMNS

for col in ["protocol_type", "service", "flag", "label"]:
    mask = df[col].apply(lambda x: isinstance(x, bytes))
    df.loc[mask, col] = df.loc[mask, col].apply(lambda x: x.decode("utf-8"))

df["label"] = df["label"].str.rstrip(".")

print(f"Dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

## 2. Dataset Overview

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Label Distribution

In [ ]:
print("Label value counts:")
print(df["label"].value_counts())

In [ ]:
df["attack_category"] = df["label"].apply(lambda x: "normal" if x == "normal" else x)
df.loc[~df["label"].isin(["normal", "buffer_overflow", "loadmodule", "rootkit", "perl", 
                          "ftp_write", "guess_passwd", "imap", "multihop", "phf", 
                          "pod", "teardrop", "land", "neptune", "smurf", "warezmaster",
                          "warezclient", "spy", "portsweep", "ipsweep", "nmap", "satan", 
                          "back"]), "attack_category"] = "other"

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

binary_labels = df["label"].apply(lambda x: "normal" if x == "normal" else "attack")
binary_labels.value_counts().plot(kind="bar", ax=axes[0], color=["#4C72B0", "#DD8452"])
axes[0].set_title("Binary Classification Distribution")
axes[0].set_ylabel("Count")

df["label"].value_counts().plot(kind="bar", ax=axes[1], color=sns.color_palette("Set2", 23))
axes[1].set_title("Multi-class Distribution")
axes[1].set_ylabel("Count")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

## 4. Categorical Features Analysis

In [ ]:
categorical_cols = ["protocol_type", "service", "flag"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, col in enumerate(categorical_cols):
    counts = df[col].value_counts().head(10)
    counts.plot(kind="bar", ax=axes[i], color=sns.color_palette("Set2", 10))
    axes[i].set_title(f"Top 10 {col}")
    axes[i].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 5. Numerical Features Distribution

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
axes = axes.flatten()

for i, col in enumerate(numerical_cols[:16]):
    df[col].plot(kind="hist", ax=axes[i], bins=50, color="#4C72B0", alpha=0.7)
    axes[i].set_title(col, fontsize=10)

plt.tight_layout()
plt.show()

## 6. Missing Values Check

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print("Missing values found:")
    print(missing)
else:
    print("No missing values in the dataset.")

## 7. Duplicate Rows

In [ ]:
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates} ({duplicates/len(df)*100:.2f}%)")

## 8. Correlation Analysis

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(14, 12))
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, square=True, 
            linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation Heatmap - All Numerical Features", fontsize=14)
plt.tight_layout()
plt.show()

## 9. Attack Type Breakdown

In [ ]:
attack_types = {
    "DoS": ["back", "land", "neptune", "pod", "smurf", "teardrop", "mailbomb", "processtable", "udpstorm"],
    "Probe": ["ipsweep", "nmap", "portsweep", "satan", "mscan", "saint"],
    "R2L": ["ftp_write", "guess_passwd", "imap", "multihop", "phf", "spy", "warezclient", "warezmaster", "named", "snmpgetattack", "worm", "xlock", "xsnoop", "sendmail", "snmpguess", "http_tunnel", "sqlattack"],
    "U2R": ["buffer_overflow", "loadmodule", "rootkit", "perl", "ps", "xterm", "tt"]
}

def categorize_attack(label):
    if label == "normal":
        return "Normal"
    for category, attacks in attack_types.items():
        if label in attacks:
            return category
    return "Other"

df["attack_category"] = df["label"].apply(categorize_attack)

category_counts = df["attack_category"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

category_counts.plot(kind="bar", ax=axes[0], color=sns.color_palette("Set2", 5))
axes[0].set_title("Attack Categories Distribution")
axes[0].set_ylabel("Count")

category_pct = category_counts / category_counts.sum() * 100
axes[1].pie(category_pct.values, labels=category_pct.index, autopct='%1.1f%%', 
            colors=sns.color_palette("Set2", 5), startangle=90)
axes[1].set_title("Attack Categories Percentage")

plt.tight_layout()
plt.show()

print("\nAttack Category Summary:")
for cat, count in category_counts.items():
    print(f"  {cat}: {count:,} ({count/len(df)*100:.1f}%)")

## 10. Key Insights Summary

In [ ]:
print("=" * 60)
print("EDA KEY INSIGHTS")
print("=" * 60)
print(f"Total samples: {len(df):,}")
print(f"Total features: {len(df.columns) - 1} (41 raw + derived)")
print(f"\nNormal samples: {(df['label'] == 'normal').sum():,}")
print(f"Attack samples: {(df['label'] != 'normal').sum():,}")
print(f"\nClass imbalance ratio: {(df['label'] != 'normal').sum() / (df['label'] == 'normal').sum():.2f}")
print(f"\nDuplicate rows: {df.duplicated().sum():,}")
print(f"\nUnique attack types: {df[df['label'] != 'label']['label'].nunique()}")
print("=" * 60)